# ClinAuthBench v1 Baseline Notebook

This notebook demonstrates how to load ClinAuthBench v1 from Hugging Face, inspect a case, build a zero-shot prompt, and compute first-pass metrics.

It is intentionally lightweight. The goal is usability and benchmark protocol, not a full leaderboard or production evaluation harness.

## 1. Install And Import

If you are running in Colab, uncomment the install line.

In [ ]:
%pip install -q datasets
import sys, os
sys.path.insert(0,os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
import json
import re
from statistics import mean

from datasets import load_dataset

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.75 requires requests_mock, which is not installed.
streamlit 1.30.0 requires packaging<24,>=16.8, but you have packaging 26.1 which is incompatible.
streamlit 1.30.0 requires protobuf<5,>=3.20, but you have protobuf 6.33.6 which is incompatible.
streamlit 1.30.0 requires tenacity<9,>=8.1.0, but you have tenacity 9.1.4 which is incompatible.
langchain-core 0.1.52 requires packaging<24.0,>=23.2, but you have packaging 26.1 which is incompatible.
langchain-core 0.1.52 requires tenacity<9.0.0,>=8.1.0, but you have tenacity 9.1.4 which is incompatible.
langchain-community 0.0.38 requires tenacity<9.0.0,>=8.1.0, but you have tenacity 9.1.4 which is incompatible.
conda-repo-cli 1.0.75 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.75 requires requests==2.31.

## 2. Load ClinAuthBench From Hugging Face

Edit `DATASET_ID` if you publish or mirror the dataset under a different Hugging Face namespace.

In [4]:
DATASET_ID = "Shivi1982/clin-auth-bench"

ds = load_dataset(DATASET_ID, split="test")
ds

README.md: 0.00B [00:00, ?B/s]

synthetic_bh_cases_v1_mdp_180.json: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/180 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'metadata', 'content'],
    num_rows: 180
})

## 3. Inspect One Case

Each record has `id`, `title`, `content`, and `metadata`. The model-facing chart packet is in `content`; gold labels are in `metadata.gold`.

In [5]:
case = ds[0]

print("Case ID:", case["id"])
print("Title:", case["title"])
print("\nFirst 1000 characters of content:\n")
print(case["content"][:1000])
print("\nGold metadata:\n")
print(json.dumps(case["metadata"]["gold"], indent=2))

Case ID: clin_auth_bench_v1_0001
Title: Current denial with high historical psychosis risk

First 1000 characters of content:

FORM: Initial Nursing Treatment Plan | CREATION_DATE: 2025-04-01T18:29:00Z
Admission: Time of Admission1829 | Admission: Date of Admission20250401 | Admission Suicide Risk Screening has been reviewed and the Overall Risk Level Score is noted as:: High | Psychiatric Problem: Depressed Mood WITH Psychosis | Depressed Mood WITH Psychosis: AEB: unsafe thoughts with intermittent command auditory hallucinations and poor sleep | Depressed Mood WITH Psychosis: GOAL: Patient will not exhibit self harm behaviors while in hospital | Depressed Mood WITH Psychosis: INTERVENTION: Query patient regarding thoughts of self-harm at least q waking shift | Depressed Mood WITH Psychosis: INTERVENTION: Encourage patient to seek staff when urges, voices, or unsafe thoughts increase | High Suicide Risk: GOAL: Patient will not attempt suicide while in hospital. | High Suicide Risk: INT

## 4. Build A Zero-Shot Prompt

The prompt asks a model to return strict JSON for a small set of benchmark fields. It also includes the case-specific `do_not_claim` rules so unsupported-claim behavior can be evaluated.

In [6]:
def build_zero_shot_prompt(record):
    gold = record["metadata"]["gold"]
    do_not_claim = gold.get("do_not_claim", [])
    do_not_claim_text = "\n".join(f"- {rule}" for rule in do_not_claim)

    return f"""
You are evaluating a synthetic inpatient health authorization chart packet.

Task: read the chart packet and return ONLY valid JSON with this schema:
{{
  "safe_for_lloc": true,
  "expected_los_recommendation": 0,
  "current_suicide_risk": "short evidence-grounded label",
  "evidence_forms": ["form names that support the answer"],
  "claims_to_avoid": ["unsupported claims the model avoided"],
  "rationale": "brief evidence-grounded explanation"
}}

Do not invent evidence. Do not rely on one favorable note if the broader packet contradicts it.

Case-specific unsupported claims to avoid:
{do_not_claim_text}

CHART PACKET:
{record["content"]}
""".strip()


prompt = build_zero_shot_prompt(ds[0])
print(prompt[:2500])

You are evaluating a synthetic inpatient health authorization chart packet.

Task: read the chart packet and return ONLY valid JSON with this schema:
{
  "safe_for_lloc": true,
  "expected_los_recommendation": 0,
  "current_suicide_risk": "short evidence-grounded label",
  "evidence_forms": ["form names that support the answer"],
  "claims_to_avoid": ["unsupported claims the model avoided"],
  "rationale": "brief evidence-grounded explanation"
}

Do not invent evidence. Do not rely on one favorable note if the broader packet contradicts it.

Case-specific unsupported claims to avoid:
- Do not confuse historical suicide risk with current risk.
- Do not conclude discharge readiness from one favorable note or a negative discharge screener alone.
- Do not invent missing or malformed rating scores.
- Do not ignore unresolved lower-level-of-care barriers.

CHART PACKET:
FORM: Initial Nursing Treatment Plan | CREATION_DATE: 2025-04-01T18:29:00Z
Admission: Time of Admission1829 | Admission: Da

## 5. Run 3-5 Sample Cases

This cell prepares a few sample prompts. Use them for manual copy/paste testing, or connect your preferred model API in the placeholder function below.

In [ ]:
SAMPLE_INDICES = [0, 120, 170]
sample_records = [ds[i] for i in SAMPLE_INDICES]

for record in sample_records:
    print("=" * 80)
    print(record["id"], "|", record["title"])
    print(build_zero_shot_prompt(record)[:1200])
    print()

In [ ]:
def call_model_placeholder(prompt):
    """Replace this function with your preferred model API call.

    Expected return type: a Python dict matching the JSON schema in the prompt.
    Keep this notebook provider-neutral for the first public baseline.
    """
    raise NotImplementedError("Add your model call here.")

# This placeholder is intentional.
# The notebook remains runnable for local parsing/scoring demonstration using mock outputs.
# For live model-output retrieval, use evals/retrieve_hf_outputs.py,evals/retrieve_openai_outputs.py, or evals/retrieve_claude_outputs.py.
# Example usage after implementing call_model_placeholder:
# predictions = []
# for record in sample_records:
#     prompt = build_zero_shot_prompt(record)
#     pred = call_model_placeholder(prompt)
#     pred["case_id"] = record["id"]
#     predictions.append(pred)

## 6. Metric Helpers

These are first-pass metric helpers:

- `safe_for_lloc` accuracy
- `expected_los_recommendation` exact match
- evidence-anchor form recall
- heuristic `do_not_claim` violation count

The `do_not_claim` helper is only a lexical proxy. Full unsupported-claim evaluation should use human review or a stronger semantic judge.

In [ ]:
def normalize_text(value):
    return re.sub(r"\s+", " ", str(value or "").strip().lower())


def safe_for_lloc_accuracy(records, predictions):
    correct = 0
    total = 0
    for record, pred in zip(records, predictions):
        if "safe_for_lloc" not in pred:
            continue
        gold = bool(record["metadata"]["gold"]["safe_for_lloc"])
        predicted = bool(pred["safe_for_lloc"])
        correct += int(gold == predicted)
        total += 1
    return correct / total if total else None


def expected_los_exact_match(records, predictions):
    correct = 0
    total = 0
    for record, pred in zip(records, predictions):
        if "expected_los_recommendation" not in pred:
            continue
        gold = normalize_text(record["metadata"]["gold"]["expected_los_recommendation"])
        predicted = normalize_text(pred["expected_los_recommendation"])
        correct += int(gold == predicted)
        total += 1
    return correct / total if total else None


def gold_evidence_forms(record):
    gold = record["metadata"]["gold"]
    forms = set(gold.get("key_evidence_forms", []))
    for anchor in gold.get("evidence_anchors", []):
        supporting_form = anchor.get("supporting_form")
        if supporting_form:
            forms.add(supporting_form)
    return forms


def evidence_anchor_form_recall(records, predictions):
    recalls = []
    for record, pred in zip(records, predictions):
        gold_forms = {normalize_text(form) for form in gold_evidence_forms(record)}
        pred_forms = {normalize_text(form) for form in pred.get("evidence_forms", [])}
        if not gold_forms:
            continue
        recalls.append(len(gold_forms & pred_forms) / len(gold_forms))
    return mean(recalls) if recalls else None


STOPWORDS = {
    "a", "an", "and", "are", "as", "be", "but", "by", "do", "does", "from", "in",
    "is", "it", "not", "of", "or", "that", "the", "to", "with", "without"
}


def important_terms(text):
    terms = re.findall(r"[a-zA-Z][a-zA-Z-]+", normalize_text(text))
    return {term for term in terms if len(term) >= 4 and term not in STOPWORDS}


def do_not_claim_violation_count(records, predictions, min_overlap=3):
    """Lexical proxy for unsupported claims.

    This checks only narrative fields where the model makes claims. It intentionally
    ignores `claims_to_avoid`, because that field may repeat the rules correctly.
    """
    violations = []
    for record, pred in zip(records, predictions):
        gold_rules = record["metadata"]["gold"].get("do_not_claim", [])
        model_claim_text = " ".join(
            str(pred.get(field, ""))
            for field in ["rationale", "authorization_summary", "predicted_claims", "current_suicide_risk"]
        )
        model_terms = important_terms(model_claim_text)
        for rule in gold_rules:
            overlap = important_terms(rule) & model_terms
            if len(overlap) >= min_overlap:
                violations.append({
                    "case_id": record["id"],
                    "rule": rule,
                    "overlap_terms": sorted(overlap),
                })
    return violations


def evaluate_predictions(records, predictions):
    violations = do_not_claim_violation_count(records, predictions)
    return {
        "n": len(predictions),
        "safe_for_lloc_accuracy": safe_for_lloc_accuracy(records, predictions),
        "expected_los_exact_match": expected_los_exact_match(records, predictions),
        "evidence_anchor_form_recall": evidence_anchor_form_recall(records, predictions),
        "do_not_claim_violation_count_heuristic": len(violations),
        "do_not_claim_violation_examples": violations[:5],
    }

## 7. Test Metrics With Mock Predictions

The predictions below are mock outputs to test the metric code. They are not model results and should not be reported as benchmark performance.

In [ ]:
mock_predictions = []

for record in sample_records:
    gold = record["metadata"]["gold"]
    mock_predictions.append({
        "case_id": record["id"],
        "safe_for_lloc": gold["safe_for_lloc"],
        "expected_los_recommendation": gold["expected_los_recommendation"],
        "current_suicide_risk": gold["current_suicide_risk"],
        "evidence_forms": gold.get("key_evidence_forms", [])[:3],
        "claims_to_avoid": gold.get("do_not_claim", []),
        "rationale": "Mock prediction for metric testing only.",
    })

metrics = evaluate_predictions(sample_records, mock_predictions)
print(json.dumps(metrics, indent=2))

# Testing For Baselining 

In [8]:
from github_release.evals.metrics import evaluate_records, evaluate_by_challenge

mock_predictions = []

for record in ds:
    gold = record["metadata"]["gold"]

    mock_predictions.append({
        "case_id": record["id"],
        "safe_for_lloc": gold["safe_for_lloc"],
        "expected_los_recommendation": gold["expected_los_recommendation"],
        "lower_level_of_care_barriers": [],
        "evidence_forms": [],
        "rationale": "Mock prediction for metric testing only."
    })

overall = evaluate_records(list(ds), mock_predictions)
by_challenge = evaluate_by_challenge(list(ds), mock_predictions)

overall, by_challenge

({'n_cases': 180,
  'n_predictions': 180,
  'usable_predictions': 171,
  'safe_for_lloc_accuracy': 1.0,
  'expected_los_exact_match': 1.0,
  'safe_for_lloc_confusion': {'gold_false_pred_false': 108,
   'gold_false_pred_true': 0,
   'gold_true_pred_false': 0,
   'gold_true_pred_true': 72}},
 {'current_vs_historical_risk': {'n_cases': 45,
   'n_predictions': 45,
   'usable_predictions': 45,
   'safe_for_lloc_accuracy': 1.0,
   'expected_los_exact_match': 1.0,
   'safe_for_lloc_confusion': {'gold_false_pred_false': 1,
    'gold_false_pred_true': 0,
    'gold_true_pred_false': 0,
    'gold_true_pred_true': 44}},
  'contradiction': {'n_cases': 22,
   'n_predictions': 22,
   'usable_predictions': 22,
   'safe_for_lloc_accuracy': 1.0,
   'expected_los_exact_match': 1.0,
   'safe_for_lloc_confusion': {'gold_false_pred_false': 22,
    'gold_false_pred_true': 0,
    'gold_true_pred_false': 0,
    'gold_true_pred_true': 0}},
  'lower_level_of_care_barrier_reasoning': {'n_cases': 81,
   'n_predict